# Comparação dos quatro métodos finais

AST original, contração Jacobiana, Shao–Wu com `tanh` nos fatores de produtos, e VAR estacionário de Heaps. O notebook 6 anterior usava `tanh` em todos os lags; seus resultados não medem o método final. Este estudo mantém um mecanismo e uma trajetória por tentativa, com ruído em todos os passos.

In [1]:
from dataclasses import asdict, replace
from pathlib import Path
from time import perf_counter
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "src" / "tsfmxai").exists())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tsfmxai.generators import (
    RandomTreeConfig, ContractionTreeConfig, ShaoWuConfig, HeapsVARConfig,
    sample_random_mechanisms, sample_contractive_mechanisms,
    sample_shao_wu_mechanisms, sample_heaps_var_mechanisms,
)
from tsfmxai.evaluation import StabilityDiagnosticConfig, wilson_interval
from tsfmxai.evaluation.generator_tradeoff import benchmark_stability_method

ROOT_SEED = 20261110
COUNT = 128
HORIZONS = (96, 192, 384)
NOISE_STD = .05
BASE = RandomTreeConfig(minimum_operators=1, maximum_operators=6, num_series=1,
                        minimum_lag=1, maximum_lag=16, parameter_probability=.55,
                        lag_probability=.45, add_probability=.8, require_lag=True,
                        minimum_parameter_magnitude=.05, maximum_parameter_magnitude=.8)
CONFIGS = {
    "original": BASE,
    "jacobian": ContractionTreeConfig(tree_config=replace(BASE, coefficient_gate_lags=True),
                                        domain_radius=1., innovation_bound=np.sqrt(3)*NOISE_STD,
                                        lag_weight_growth=1.25, target_lipschitz=.9,
                                        scale_safety_factor=.95),
    "shao-wu-product-lags": ShaoWuConfig(tree_config=BASE, innovation_std=NOISE_STD,
                                           target_gain=.8),
    "heaps-var": HeapsVARConfig(num_series=1, minimum_order=1, maximum_order=12,
                                 unconstrained_std=.45, innovation_std=NOISE_STD),
}
DIAGNOSTICS = StabilityDiagnosticConfig(HORIZONS, 100., 3*NOISE_STD, 24)
SAMPLERS = {
    "original": sample_random_mechanisms,
    "jacobian": sample_contractive_mechanisms,
    "shao-wu-product-lags": sample_shao_wu_mechanisms,
    "heaps-var": sample_heaps_var_mechanisms,
}
results = {}
for method, config in CONFIGS.items():
    start = perf_counter()
    mechanisms = SAMPLERS[method](config, root_seed=ROOT_SEED, count=COUNT)
    sampling_seconds = perf_counter() - start
    if method != "original":
        assert all(m.certificate.holds for m in mechanisms)
    results[method] = benchmark_stability_method(
        method, mechanisms, root_seed=ROOT_SEED, diagnostic_config=DIAGNOSTICS,
        innovation_std=NOISE_STD, sampling_seconds=sampling_seconds, keep_examples=0)
    print(method, "completed", flush=True)

outcomes = pd.DataFrame(row for result in results.values() for row in result["outcomes"])
summary = outcomes.groupby(["method", "horizon"], sort=False).agg(
    attempts=("sample_index", "size"), explosions=("exploded", "sum"),
    near_zero=("near_zero", "sum"), nonfinite=("nonfinite", "sum"),
    finite_threshold_crossings=("finite_threshold_crossing", "sum"),
).reset_index()
assert summary.attempts.eq(COUNT).all()
summary["explosion_rate"] = summary.explosions / summary.attempts
summary["near_zero_rate"] = summary.near_zero / summary.attempts
summary["explosion_wilson_95"] = [wilson_interval(int(k), int(n))
                                  for k, n in zip(summary.explosions, summary.attempts)]
performance = pd.DataFrame(result["performance"] for result in results.values())
display(summary)
display(performance)


original completed


jacobian completed


shao-wu-product-lags completed


heaps-var completed


,method,horizon,attempts,explosions,near_zero,nonfinite,finite_threshold_crossings,explosion_rate,near_zero_rate,explosion_wilson_95
0,original,96,128,52,14,0,52,0.406250,0.109375,"(0.3251030219432171, 0.4928601573580657)"
1,original,192,128,57,15,0,57,0.445312,0.117188,"(0.36205524949299805, 0.5317566050994169)"
2,original,384,128,63,18,0,63,0.492188,0.140625,"(0.4070774941965585, 0.5777527707452151)"
3,jacobian,96,128,0,69,0,0,0.000000,0.539062,"(0.0, 0.029136956273508416)"
4,jacobian,192,128,0,71,0,0,0.000000,0.554688,"(0.0, 0.029136956273508416)"
5,jacobian,384,128,0,71,0,0,0.000000,0.554688,"(0.0, 0.029136956273508416)"
6,shao-wu-product-lags,96,128,0,26,0,0,0.000000,0.203125,"(0.0, 0.029136956273508416)"
7,shao-wu-product-lags,192,128,0,28,0,0,0.000000,0.218750,"(0.0, 0.029136956273508416)"
8,shao-wu-product-lags,384,128,0,32,0,0,0.000000,0.250000,"(0.0, 0.029136956273508416)"
9,heaps-var,96,128,0,43,0,0,0.000000,0.335938,"(0.0, 0.029136956273508416)"


,method,mechanisms,sampling_and_certificate_seconds,mechanisms_per_second,initialization_seconds,innovation_seconds,ast_execution_seconds,diagnostic_seconds,evaluation_seconds,total_seconds,attempted_trajectories_per_second,successful_full_trajectories_per_second,full_horizon_success_rate,evaluated_steps,evaluated_values,ast_values_per_second,end_to_end_values_per_second
0,original,128,0.041062,3117.275300,0.011838,0.004357,0.526312,0.009998,0.562084,0.603146,212.220731,107.768340,0.507812,29229,29229,55535.468175,48460.935464
1,jacobian,128,0.081594,1568.737032,0.015686,0.004017,1.065944,0.013428,1.109576,1.191170,107.457365,107.457365,1.000000,49152,49152,46111.244665,41263.628091
2,shao-wu-product-lags,128,0.099239,1289.820695,0.007067,0.004609,0.990152,0.011544,1.023836,1.123074,113.972859,113.972859,1.000000,49152,49152,49640.873237,43765.577774
3,heaps-var,128,0.370482,345.495588,0.011279,0.007512,1.291727,0.011710,1.333331,1.703814,75.125589,75.125589,1.000000,49152,49152,38051.392766,28848.226120


## Registro completo do experimento

A célula seguinte contém o desenho, a procedência, as sementes, as métricas e a reprodução.

In [2]:
source_paths = [
    "src/tsfmxai/generators/random_expression.py",
    "src/tsfmxai/generators/contraction.py",
    "src/tsfmxai/generators/shao_wu.py",
    "src/tsfmxai/generators/heaps.py",
    "src/tsfmxai/evaluation/generator_tradeoff.py",
    "src/tsfmxai/evaluation/stability.py",
]
def git_head():
    return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT,
                                   text=True).strip()

record = {
    "experiment_id": "abstract_syntax_tree_final_methods_20261110",
    "status": "executed",
    "research_question": "Como os quatro métodos finais diferem em explosão, quase zero e custo até 384 passos?",
    "hypothesis": "Os métodos certificados terão menos falhas observadas que a AST original, com custo de amostragem maior.",
    "configuration": {
        "count_per_method": COUNT, "num_series": 1, "trajectories_per_mechanism": 1,
        "diagnostics": asdict(DIAGNOSTICS), "innovation_std": NOISE_STD,
        "methods": {name: asdict(config) for name, config in CONFIGS.items()},
        "initialization": "original: uniform [-1,1]; Jacobian: uniform in certified box; Shao-Wu: direct uniform [-1,1]; Heaps: stationary joint Gaussian",
        "innovation": "Gaussian for original/Shao-Wu/Heaps; bounded uniform for Jacobian",
        "failure_rule": "nonfinite or |x| > 100, as an operational diagnostic, not a sampler rejection",
        "timing": "serial wall-clock, one measurement; do not compare as a hardware-independent speed claim",
    },
    "seed_policy": {"root_seed": ROOT_SEED,
                    "mechanisms": "derive_sample_seed(ROOT_SEED, index)",
                    "evaluation": "SeedSequence([ROOT_SEED, index, 991]).spawn(2)"},
    "provenance": {"git_head": git_head(),
                   "dirty_worktree_at_execution": bool(subprocess.check_output(
                       ["git", "status", "--porcelain"], cwd=PROJECT_ROOT)),
                   "python": sys.version,
                   "platform": platform.platform(),
                   "packages": {p: importlib.metadata.version(p) for p in ("numpy", "pandas", "nbformat", "nbclient")},
                   "source_sha256": {p: hashlib.sha256((PROJECT_ROOT/p).read_bytes()).hexdigest()
                                     for p in source_paths}},
    "metrics": {"summary": summary.to_dict(orient="records"),
                "performance": performance.to_dict(orient="records")},
    "scientific_summary": (
        f"Em {COUNT} tentativas por método até {max(HORIZONS)} passos, "
        + "; ".join(f"{row.method}: {row.explosions} explosões e {row.near_zero} quase zero"
                    for row in summary[summary.horizon.eq(max(HORIZONS))].itertuples())
        + ". São observações finitas. Um certificado individual não prova estabilidade "
          "de todo acoplamento futuro. O limiar operacional de 100 é um diagnóstico, não um filtro."
    ),
    "limitations": "128 mecanismos por família dão intervalos amplos; velocidade em uma máquina; métodos têm distribuições e inovações diferentes.",
    "reproduction": "Na raiz: .venv/Scripts/python -m jupyter nbconvert --to notebook --execute experiments/abstract_syntax_tree/final_methods/00_final_methods_tradeoff.ipynb --output 00_final_methods_tradeoff.ipynb --output-dir experiments/abstract_syntax_tree/final_methods",
}
print(json.dumps(record, ensure_ascii=False, indent=2, default=lambda v: float(v) if isinstance(v, np.floating) else int(v) if isinstance(v, np.integer) else list(v)))


{
  "experiment_id": "abstract_syntax_tree_final_methods_20261110",
  "status": "executed",
  "research_question": "Como os quatro métodos finais diferem em explosão, quase zero e custo até 384 passos?",
  "hypothesis": "Os métodos certificados terão menos falhas observadas que a AST original, com custo de amostragem maior.",
  "configuration": {
    "count_per_method": 128,
    "num_series": 1,
    "trajectories_per_mechanism": 1,
    "diagnostics": {
      "horizons": [
        96,
        192,
        384
      ],
      "explosion_limit": 100.0,
      "near_zero_radius": 0.15000000000000002,
      "near_zero_tail_steps": 24
    },
    "innovation_std": 0.05,
    "methods": {
      "original": {
        "minimum_operators": 1,
        "maximum_operators": 6,
        "num_series": 1,
        "minimum_lag": 1,
        "maximum_lag": 16,
        "parameter_probability": 0.55,
        "lag_probability": 0.45,
        "add_probability": 0.8,
        "minimum_parameter_magnitude": 0.05,
  